# 7b · N-ary relation canonicalization  (twin of nb7, for the reified graph)

The property-extension stage for the **event-centric** path. It is nb7's twin: same machinery
(embedding retrieval → one LLM call that matches each free-form `relation_phrase` to a schema.org OR
vocabulary property, else mints a new one → a growing vocabulary), but pointed at the **open n-ary
graph** from nb6b instead of the binary triples.

Differences from nb7:
- **Input:** reads `results/<id>/nary_graph.json` (nb6b) — n-ary nodes + open `relations`.
- **Endpoints:** a relation's subject/object may be an ENTITY (`E..`) or an **n-ary node**
  (`EV../OBS../CL..`); both resolve to a type for the domain→range signatures.
- **Separate vocabulary:** grows `results/nary_relation_vocabulary.json` — kept independent of the
  binary path's `relation_vocabulary.json`, so the two methods' vocabularies stay comparable.
- **Output:** writes `results/<id>/nary_graph_ext.json` (the n-ary graph with each relation
  canonicalized: `property`, `property_source ∈ {schema.org, vocab}`, `vocab_id`,
  `property_is_new_to_vocab`, `property_definition`, `property_parent`, `relation_phrases`, …) plus a
  `nary_relation_matching_review.json` audit file. `nary_nodes` and `entities` are carried through.

Point nb8's `REL_FILES` at `nary_graph_ext.json` to see the canonicalized graph (schema.org solid /
vocab dashed). The optional GLOBAL re-consolidation + property-hierarchy passes from nb7 port over
unchanged (operate on the vocab + ext files) — add them here if/when you want that cleanup.

In [1]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [2]:
# ---- WHICH MODEL'S RESULTS ---------------------------------------------------------------
# Everything is read from and written to results/<RESULTS_MODEL>/. Choose any subfolder that
# exists there: "gemini-2.5-flash", "qwen3-14b", "qwen3-32b", "llama33-70b", ... (the names are
# the BACKENDS keys in notebooks/llm_backend.py).
RESULTS_MODEL = "gemini-2.5-flash"

# Which model runs THIS notebook's LLM calls. Usually the same one; set it to
# "gemini-2.5-flash" to process another model's outputs with Gemini (e.g. as an eval judge).
# Anything else needs a vLLM server + tunnel — see cluster/README.md.
BACKEND = RESULTS_MODEL
# ------------------------------------------------------------------------------------------

# --- Config: n-ary relation canonicalization (nb7b) ----------------------------------------------------
import json, re, os, copy
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

SCHEMA_DIR  = ROOT / "schema_org_files"     # property pool from nb5
RESULTS_DIR = ROOT / "results" / RESULTS_MODEL   # reads nary_graph.json (nb6b)
VOCAB_PATH  = RESULTS_DIR / "_global/nary_relation_vocabulary.json"   # SEPARATE from the binary path

MATCH_MODEL  = "intfloat/e5-base-v2"       # ONE embedding model for the whole vocabulary (don't mix spaces)
GEMINI_MODEL = "gemini-2.5-flash"
EXPAND_MODE  = True                        # True: admit minted properties (grow). False: restrict to schema.org u vocab.

# No mint-time canonicalization: a minted property is admitted as-is during the incremental build.
# Collapsing synonymous / broader-narrower mints is deferred to the OPTIONAL global re-consolidation
# pass at the bottom (consolidate_vocab) — run it to clean up, or skip it to keep the raw vocabulary.

TOP_K_MATCH  = 10                          # candidate properties shown to the selection LLM (Hook A)
ALIGN_QUERY_STYLE = True                   # append "Also called <synonyms>." to each relation-phrase query

# Which articles nb7 runs on — and the order the vocabulary grows along (order matters: incremental).
#   None  -> every article that has a nary_graph.json (from nb6b), sorted.
#   list  -> exactly these ids, in this order, e.g. ["art2", "art1", "art20"].
#SELECTED_ARTICLES = None
SELECTED_ARTICLES = ["art1", "art2", "art4", "art5"]

_available = sorted(p.relative_to(RESULTS_DIR).parts[0] for p in RESULTS_DIR.glob("*/two_staged/3_nary_graph/nary_graph.json"))
if SELECTED_ARTICLES is None:
    ARTICLE_ORDER = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    if missing:
        print("WARNING: skipping (no nary_graph.json):", missing)
    ARTICLE_ORDER = [a for a in SELECTED_ARTICLES if a in _available]
assert ARTICLE_ORDER, "No runnable articles selected (need nb6b nary_graph.json). Check SELECTED_ARTICLES."
print("expand:", EXPAND_MODE, "| match:", MATCH_MODEL, "| articles:", ARTICLE_ORDER)

expand: True | match: intfloat/e5-base-v2 | articles: ['art1', 'art2', 'art4', 'art5']


In [3]:
# First run only (sentence-transformers pulls torch; google-genai for the LLM stages).
%pip install -q sentence-transformers google-genai numpy

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Embedding model (loaded once). e5 needs "query: " on the query side and "passage: " on documents;
# both the relation-phrase queries and every document (schema.org OR vocab property) go through the
# same convention — identical to nb4.
from sentence_transformers import SentenceTransformer
_MODEL = SentenceTransformer(MATCH_MODEL)

def _prefixes(name):
    n = name.lower()
    if "e5" in n:  return "query: ", "passage: "
    if "bge" in n: return "Represent this sentence for searching relevant passages: ", ""
    return "", ""
QP, DP = _prefixes(MATCH_MODEL)

def embed(texts, prefix):
    return np.asarray(_MODEL.encode([prefix + t for t in texts],
                                    normalize_embeddings=True, batch_size=64))
print("embedder ready:", MATCH_MODEL, "| query_prefix", repr(QP), "| doc_prefix", repr(DP))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedder ready: intfloat/e5-base-v2 | query_prefix 'query: ' | doc_prefix 'passage: '


In [5]:
# Schema.org PROPERTY document pool (fixed) — embedded once (from nb5). Vocabulary properties are
# embedded per article inside retrieval, since the vocabulary grows during the run.
schema_pool   = json.loads((SCHEMA_DIR / "schema_properties_enriched.json").read_text())
SCHEMA_LABELS = {p["label"] for p in schema_pool}
schema_emb    = embed([p["embedding_text"] for p in schema_pool], DP)
print(len(schema_pool), "schema.org properties embedded")

1521 schema.org properties embedded


In [6]:
# The growing property vocabulary store. One entry per canonical minted property:
#   id, canonical_label (camelCase property name), definition, parent (subPropertyOf: a schema.org OR
#   vocab property label, or null), domain[] / range[] (subject/object type hints), aliases[] (its
#   synonyms), provenance[], support, first_seen, and path (derived: super-property chain, stamped on save).
def load_vocab():
    if VOCAB_PATH.exists():
        return json.loads(VOCAB_PATH.read_text())
    return {"properties": [], "next_id": 1}

def save_vocab(vocab):
    _annotate_vocab_paths(vocab)                 # refresh each entry's derived `path` from its parent chain
    VOCAB_PATH.write_text(json.dumps(vocab, indent=2, ensure_ascii=False))

def _sig(domain, range_):
    """domain → range for HUMAN display / the LLM judge prompt (the '→' glyph is fine there). Kept
    OUT of embedding_text — see vocab_embedding_text."""
    d = "/".join(domain) if domain else "ANY"
    r = "/".join(range_) if range_ else "ANY"
    return f"{d} → {r}"

def vocab_embedding_text(t):
    # "label: definition Also called <aliases>." — the SAME style as nb7's relation-phrase query and
    # nb5's schema property docs, so query and both document sources share one embedding style. The
    # domain -> range signature is intentionally not embedded (see nb5); it goes to the LLM judge only.
    also = f" Also called {', '.join(t['aliases'])}." if t.get("aliases") else ""
    return f'{t["canonical_label"]}: {t.get("definition", "")}{also}'

def _find_vocab_by_label(vocab, label):
    for t in vocab["properties"]:
        if t["canonical_label"] == label or label in t.get("aliases", []):
            return t
    return None

def _is_vocab_label(vocab, label):
    return _find_vocab_by_label(vocab, label) is not None

def _bump(ent, aid, phrase, instances, flip=False):
    # provenance instances are oriented to the PROPERTY's direction (swapped when flip=True), so they
    # read the same way as the stored triples.
    ent["support"] = ent.get("support", 0) + len(instances)
    def _pair(i):
        return f'{i["object_text"]} -> {i["subject_text"]}' if flip else f'{i["subject_text"]} -> {i["object_text"]}'
    ent.setdefault("provenance", []).append(
        {"article": aid, "relation_phrase": phrase, "direction": ("inverse" if flip else "as_is"),
         "instances": [_pair(i) for i in instances]})

In [7]:
# Hook A — retrieve top-k candidate properties for each relation phrase against schema.org u current
# vocabulary. Both sources expose their domain->range `signature` and their `subPropertyOf` parent, so
# the selection LLM can judge direction/domain-range fit and reach for either as a mint parent.
_SCHEMA_BY_LABEL = {p["label"]: p for p in schema_pool}

def _schema_super(label):
    """Immediate super-property (subPropertyOf) of a schema.org property, or None. schema.org lists
    sub_property_of as a (usually empty) list; we take the first schema.org super-property."""
    p = _SCHEMA_BY_LABEL.get(label)
    for sup in (p.get("sub_property_of") if p else []) or []:
        if sup in SCHEMA_LABELS:
            return sup
    return None

def _schema_super_chain(label):
    """Top-down super-property chain INCLUDING `label`: [root_super, ..., label]. Sparse for
    properties (most schema.org properties have no super-property), so usually just [label]."""
    chain, cur, seen = [], label, set()
    while cur and cur not in seen:
        seen.add(cur); chain.append(cur); cur = _schema_super(cur)
    return list(reversed(chain))

def vocab_ancestor_path(t, vocab):
    """super-property chain for a vocab property (top-down, ending at the immediate parent, EXCLUDING
    the property itself). Climbs `parent` (subPropertyOf) links up through the vocabulary, then splices
    in the schema.org super-chain of the first schema property it reaches. A `seen` guard prevents loops."""
    up, cur, seen = [], t.get("parent"), set()
    while cur and cur not in seen:
        seen.add(cur)
        ent = _find_vocab_by_label(vocab, cur)
        if ent:                                   # another vocab property -> keep climbing the vocab tree
            up.append(cur); cur = ent.get("parent")
        else:                                     # a schema.org property -> splice its super-chain in
            up.extend(reversed(_schema_super_chain(cur))); cur = None
    return " > ".join(reversed(up))

def _annotate_vocab_paths(vocab):
    """Stamp each entry's derived `path` (super-property chain, ending at the immediate parent).
    Derived from the `parent` chains (the source of truth) — refreshed on every save."""
    for t in vocab["properties"]:
        t["path"] = vocab_ancestor_path(t, vocab)
    return vocab

def _full_path(tref, vocab):
    """Full super-property path for a resolved property reference (used in the review's chosen block)."""
    if tref.get("source") == "vocab" and tref.get("vocab_id"):
        ent = next((t for t in vocab["properties"] if t["id"] == tref["vocab_id"]), None)
        if ent is not None:
            return vocab_ancestor_path(ent, vocab)
    if tref.get("source") == "schema.org" and tref.get("label"):
        return " > ".join(_schema_super_chain(tref["label"])[:-1])   # super-properties only
    return None

def retrieve_candidates(phrase_texts, vocab):
    q_emb = embed(phrase_texts, QP)
    meta  = [("schema.org", p) for p in schema_pool]
    d_emb = schema_emb
    if vocab["properties"]:
        v_emb = embed([vocab_embedding_text(t) for t in vocab["properties"]], DP)
        d_emb = np.vstack([schema_emb, v_emb])
        meta  = meta + [("vocab", t) for t in vocab["properties"]]
    sims = q_emb @ d_emb.T
    topk = np.argsort(-sims, axis=1)[:, :TOP_K_MATCH]
    out = []
    for ti in range(len(phrase_texts)):
        row = []
        for j in topk[ti]:
            src, obj = meta[j]
            if src == "schema.org":
                row.append({"source": "schema.org", "label": obj["label"], "signature": obj.get("signature", ""),
                            "parent": _schema_super(obj["label"]), "inverse_of": obj.get("inverse_of"),
                            "score": round(float(sims[ti, j]), 3),
                            "comment": obj.get("enriched_comment") or obj.get("comment", "")})
            else:
                row.append({"source": "vocab", "id": obj["id"], "label": obj["canonical_label"],
                            "signature": _sig(obj.get("domain"), obj.get("range")), "parent": obj.get("parent"),
                            "inverse_of": None,
                            "score": round(float(sims[ti, j]), 3), "comment": obj.get("definition", "")})
        out.append(row)
    return out

In [8]:
# Article inputs + Gemini client (JSON mode, temperature 0 for reproducibility).
def load_article(aid):
    """Read nb6b's nary_graph.json and group relation instances by their free-form relation_phrase.
    `by_id` resolves BOTH entities (E..) and n-ary nodes (EV../OBS../CL..) to a type, so the
    subject->object signatures work for n-ary-node endpoints too."""
    g = json.loads((RESULTS_DIR / aid / "two_staged/3_nary_graph/nary_graph.json").read_text())
    by_id = {e["id"]: {"id": e["id"], "text": e.get("text"), "type": e.get("type")}
             for e in g.get("entities", [])}
    for n in g.get("nary_nodes", []):                      # n-ary nodes are first-class endpoints
        by_id[n["id"]] = {"id": n["id"], "text": n.get("label"),
                          "type": n.get("type") or n.get("node_kind")}
    rinfo   = {d["relation_phrase"]: {"definition": d.get("definition")}
               for d in g.get("relation_definitions", [])}
    phrases = list(dict.fromkeys(r["relation_phrase"] for r in g["relations"]))
    def qtext(ph):
        info = rinfo.get(ph, {})
        q = f'{ph}: {info.get("definition", "")}'.strip()
        if ALIGN_QUERY_STYLE and info.get("synonyms"):     # nary graph has no synonyms -> no-op, kept for parity
            q = f'{q} Also called {", ".join(info["synonyms"])}.'
        return q
    groups = {}
    for r in g["relations"]:
        groups.setdefault(r["relation_phrase"], []).append(r)
    return g, by_id, rinfo, phrases, [qtext(p) for p in phrases], groups

def _etype(by_id, eid):
    return (by_id.get(eid) or {}).get("type")

def group_signatures(instances, by_id):
    """Distinct observed subject_type -> object_type signatures within a relation-phrase group."""
    sigs = Counter(f'{_etype(by_id, i["subject_id"])} → {_etype(by_id, i["object_id"])}' for i in instances)
    return [s for s, _ in sigs.most_common()]

from google import genai
from google.genai import types as genai_types
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (or GOOGLE_API_KEY) in the env and restart the kernel."
client = genai.Client(api_key=GEMINI_API_KEY)

def gemini_json(system, user):
    """One JSON-mode call. Thinking OFF so the WHOLE output budget goes to the JSON answer — gemini-2.5-flash
    dynamic thinking otherwise eats the budget and truncates the JSON mid-string on big articles."""
    cfg = dict(system_instruction=system, temperature=0, response_mime_type="application/json")
    if hasattr(genai_types, "ThinkingConfig"):
        cfg["thinking_config"] = genai_types.ThinkingConfig(thinking_budget=-1)
    import time
    t0 = time.perf_counter()
    resp = client.models.generate_content(
        model=GEMINI_MODEL, contents=user, config=genai_types.GenerateContentConfig(**cfg))
    global LAST_USAGE
    LAST_USAGE = nb_parallel.record_gemini_usage(resp, time.perf_counter() - t0)   # prompt/output/THINKING tokens + latency
    return resp

def parse_json(resp):
    """Parse Gemini JSON from a response (or raw text), tolerating fences / trailing commas, with a
    CLEAR message on truncation instead of a cryptic JSONDecodeError."""
    text = resp if isinstance(resp, str) else (getattr(resp, "text", "") or "")
    s = text.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z]*\s*", "", s); s = re.sub(r"\s*```$", "", s).strip()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        pass
    s2 = re.sub(r",(\s*[}\]])", r"\1", s)                 # drop trailing commas before } or ]
    try:
        return json.loads(s2)
    except json.JSONDecodeError as e:
        fr = None
        try: fr = resp.candidates[0].finish_reason
        except Exception: pass
        raise ValueError(f"Gemini JSON unparseable (finish_reason={fr}; {len(s)} chars). If MAX_TOKENS, "
                         f"the output was TRUNCATED — thinking is already off; split the article if it recurs. "
                         f"json error: {e}\n...tail:\n{s[-300:]}") from None


# --- select the backend ---------------------------------------------------------------------
# Must stay LAST in this cell: the Gemini `gemini_json` above is redefined every time the cell
# runs, so whichever binding happens last wins. In a separate cell this would be silently undone.
# activate() sets gemini_json, GEMINI_MODEL, RESULTS_DIR, MAX_OUTPUT_TOKENS and THINKING_BUDGET.
import sys, pathlib, importlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "llm_backend.py").exists():
        sys.path.insert(0, str(_p)); break
import nb_parallel, llm_backend
importlib.reload(nb_parallel); importlib.reload(llm_backend)   # plain `import` is a no-op once cached
nb_parallel.install()           # thread-aware stdout + the token/latency ledger used by usage_report

llm_backend.activate(BACKEND, ROOT, globals(), results=RESULTS_MODEL)


## Inspect retrieval — top-k candidates by embedding similarity (no LLM)

Preview **Hook A on its own**: for one article, show the top-`TOP_K_MATCH` candidate properties each
relation phrase retrieves from **schema.org ∪ current property-vocabulary**, ranked by cosine
similarity — *before* the LLM judges or mints anything. On a fresh run the vocabulary is empty, so the
candidates are schema.org-only; re-run this cell after **Build** (below) to see minted `ex:` properties
surface as candidates too. No Gemini call — pure retrieval.

In [9]:
# Retrieval-only preview: the ranked candidates per relation phrase, with NO Gemini call. Uses the
# CURRENT vocab state (load_vocab(); empty => schema.org-only). This is exactly the `cands` that would
# be handed to the LLM judge in process_article — inspect it first to sanity-check what the embedder
# surfaces. Set INSPECT_ARTICLE to any id in ARTICLE_ORDER.
INSPECT_ARTICLE = ARTICLE_ORDER[0]

def preview_retrieval(aid, vocab=None, top_k=TOP_K_MATCH, show_comment=True):
    vocab = load_vocab() if vocab is None else vocab
    oie, by_id, rinfo, phrases, qtexts, groups = load_article(aid)
    cands = retrieve_candidates(qtexts, vocab)
    print(f"{aid}: {len(phrases)} relation phrases | vocab has {len(vocab['properties'])} minted "
          f"property(ies)\n" + "=" * 92)
    for i, ph in enumerate(phrases):
        sigs = ", ".join(group_signatures(groups[ph], by_id))
        print(f'\n"{ph}"   ({len(groups[ph])} instance(s) | observed {sigs})')
        print(f'   query: {qtexts[i]}')
        for k, c in enumerate(cands[i][:top_k], 1):
            inv = f'   inverse: {c["inverse_of"]}' if c.get("inverse_of") else ""
            print(f'   {k}. [{c["source"]:9}] {c["label"]:<26} score={c["score"]:.3f}   sig: {c.get("signature","")}{inv}')
            if show_comment and c.get("comment"):
                print(f'          {c["comment"][:100]}')
    return cands

_ = preview_retrieval(INSPECT_ARTICLE)

art1: 44 relation phrases | vocab has 43 minted property(ies)

"has_quantity"   (7 instance(s) | observed Excess Deaths Report → QuantitativeValue, Drowning Deaths Report → StatisticalPopulation, Wildfire Incident Report → IncidentCount, Wildfire Evacuation → StatisticalPopulation)
   query: has_quantity: The subject has the specified quantitative value.
   1. [schema.org] value                      score=0.844   sig: MonetaryAmount/PropertyValue/QuantitativeValue → Boolean/Number/StructuredValue/Text
          A value relation between a monetary amount, property value, or quantitative value and its specific n
   2. [vocab    ] hasAreaValue               score=0.833   sig: Area Burned → AreaMeasurement
          The subject has the specified area value.
   3. [schema.org] eligibleQuantity           score=0.823   sig: Demand/Offer/PriceSpecification → QuantitativeValue
          An eligible quantity relation between a demand, offer, or price specification and the quantitative v
   4. [v

In [10]:
# ==== EDIT ME — prompt ===================================================================
# PROPERTY_SELECT_PROMPT is the property-side twin of nb4's TYPE selection task: canonicalize each
# free-form relation phrase to a schema.org OR vocabulary property, else mint a new ex: property.
# (Relation-instance dedup is deterministic — collapsing identical subject/object pairs onto one
# triple — so it is done in code, not by the LLM; this prompt only chooses the property.)
PROPERTY_SELECT_PROMPT = """You are an expert knowledge-engineering assistant. You are given several groups of relation instances from one document. Each group shares a single free-form relation phrase (an OIE relation label). For each group independently, assign every instance a final property — do NOT merge or compare instances across different groups.

Each group provides:
- RELATION_PHRASE: the free-form relation label these instances were given, plus its definition. A relation reads subject --RELATION_PHRASE--> object.
- OBSERVED_SIGNATURES: the subject-type -> object-type signatures actually seen among the instances (helps you check domain/range fit and direction).
- CANDIDATES: a ranked list of candidate properties proposed as matches. Each candidate is either an existing Schema.org property or an existing property from our extended vocabulary; its source is marked [schema.org] or [vocab]. Each shows its `signature` (domain -> range, i.e. the subject types it applies to and the object types it accepts) and its `parent` (its super-property via subPropertyOf, if any).
- INSTANCES: a NUMBERED list of the actual subject --> object pairs (with their entity types) that were assigned this relation phrase.

Split each group into one or more ASSIGNMENTS and choose the best property for each.
- An assignment is a subset of the group's instances (referenced by their instance NUMBERS) plus the single property chosen for that subset. Every instance MUST belong to EXACTLY ONE assignment.
- By default put the whole group in ONE assignment (one property for all its instances). Split into separate assignments ONLY when different instances genuinely fit different properties — typically because their subject/object types differ (see OBSERVED_SIGNATURES), so a different candidate fits each subset better. Do NOT split arbitrarily: instances that fit the same property stay together.
- Choose each assignment's property using the rules below (they apply to that assignment's instances):
- Prefer an existing candidate. Pick the one whose meaning (from its comment) best fits the relation phrase and the instances. What matters most is that the candidate captures the SAME relation meaning; treat the domain -> range signature as a SOFT compatibility signal, not a strict filter (see the instance test). When a Schema.org candidate and a vocabulary candidate both fit equally well, prefer the schema.org one (schema.org-first). Reusing an existing property is strongly preferred over minting.
- Direction & inverse reuse (important — this reduces redundant new properties): candidate comments are written neutrally and may not state a direction, so judge direction from the signatures (each candidate's domain -> range vs the OBSERVED_SIGNATURES) and the instance types — not from the comment wording. The stored triple MUST read subject --> object in the chosen property's own direction. A candidate fits if either:
    (a) it already runs subject --> object with our arguments — set "direction" to "as_is"; or
    (b) it expresses the inverse relation, object --> subject (its domain broadly aligns with our object type and its range with our subject type). In that case you should still reuse it rather than mint a redundant new property: select it and set "direction" to "inverse". Choosing "inverse" means we will swap subject and object so the stored triple matches the property's direction.
  A candidate may also list `inverse: <name>` — the schema.org property for the opposite direction; if that inverse property is itself the best fit, prefer selecting it directly ("as_is") when it appears in the list. Only mint when neither the direct nor the inverse reading of any candidate fits.
- Instance test — judge by the INSTANCES, not the phrase definition alone: does the candidate express the same relation meaning for these subject/object pairs, in the orientation you chose (as_is: subject against the candidate's domain, object against its range; inverse: subject against its range, object against its domain)? Treat domain/range LOOSELY here: schema.org's domain/range are permissive suggestions, and our entity types are often subtypes of, supertypes of, or siblings of a property's declared domain/range. Accept the candidate whenever it preserves the relation's meaning and the arguments are broadly compatible — do NOT reject it just because a type is more specific, more general, or a near-neighbour of the declared domain/range. Reject on domain/range grounds ONLY when the pairing is genuinely incompatible: meaningless, or a different KIND of relation that would change what the property asserts. A candidate that is somewhat broader but still preserves what makes the relation distinctive is preferable to a new property.
- Mint a new property ONLY when no existing candidate captures the relation's meaning in either direction — i.e. when reusing the best candidate would change or lose the relation's meaning (including when it is only "correct" by being so broad it loses the relation's distinguishing meaning). Do NOT mint merely because the subject/object types are not an exact match for a candidate's declared domain/range: a meaning-preserving candidate with a loose type fit is preferable to a new property. When you do mint, use a concise, GENERAL, reusable camelCase property label that names the RELATION ITSELF and does NOT embed a subject or object (neither its type nor a specific entity) in the label (write "after", not "afterEvent"; "locatedIn", not "locatedInCity"; "hasValue", not "hasQuantityValue"); the object belongs in the object slot, not the predicate, and two phrasings of the same relation must yield the SAME label; give it a one-sentence definition of the relation; a mint is always in the phrase's own direction, so its "direction" is "as_is".
- When you mint, also assign: a parent (the closest existing property this should be a subPropertyOf — a Schema.org or [vocab] candidate label, or any candidate's `parent`; use null if none is a sensible super-property), and domain / range (short lists of the subject types and object types the minted property applies to, taken from the OBSERVED_SIGNATURES).
- Report each assignment as: "instances" (the list of instance NUMBERS it covers) and "chosen_property" with: source ("schema.org" or "vocab" when you chose a candidate, matching that candidate's marked source; "new" when you mint), the chosen label (the exact candidate label, or the new camelCase label), whether it is new, its "direction" ("as_is" or "inverse"), its definition / parent / domain / range (only when new; else null), and a one-line justification (mention the flip when direction is "inverse").

Return ONLY valid JSON — no prose, no code fences. Output one result object per input group, in the same order, each echoing its exact relation_phrase string. Angle-bracket placeholders show the shape, not literal values:
{"results": [{"relation_phrase": "<phrase>", "assignments": [{"instances": [1, 2], "chosen_property": {"source": "schema.org | vocab | new", "label": "<property label>", "is_new": false, "direction": "as_is", "definition": null, "parent": null, "domain": null, "range": null, "justification": "one short line"}}]}]}
Include every group exactly once, and cover every instance NUMBER of that group exactly once across its assignments."""
print("prompt loaded")

prompt loaded


In [11]:
# Hook A caller — ONE Gemini call per article. For each relation-phrase group the LLM returns one or
# more ASSIGNMENTS: each assignment is a subset of the group's instances (by 1-based number) plus the
# property chosen for that subset. This lets a single phrase split across several properties when its
# instances have different signatures. Returns: phrase -> [{"instances": [0-based idx...], "chosen": {...}}].
def llm_assign(phrases, rinfo, groups, by_id, cands):
    blocks = []
    for i, ph in enumerate(phrases):
        b = [f"GROUP {i+1} —", f'RELATION_PHRASE: "{ph}"']
        if rinfo.get(ph, {}).get("definition"):
            b.append(f'DEFINITION: {rinfo[ph]["definition"]}')
        b.append("OBSERVED_SIGNATURES: " + "; ".join(group_signatures(groups[ph], by_id)))
        b.append("CANDIDATES (ranked):")
        if cands[i]:
            for k, c in enumerate(cands[i], 1):
                par = c.get("parent") or "—"
                inv = f' | inverse: {c["inverse_of"]}' if c.get("inverse_of") else ""
                b.append(f'  {k}. [{c["source"]}] {c["label"]} | signature: {c.get("signature","")} | subPropertyOf: {par}{inv}')
                b.append(f'     {c.get("comment", "")}')
        else:
            b.append("  (none)")
        b.append("INSTANCES (numbered):")
        b += [f'  {j}. "{r["subject_text"]}" ({_etype(by_id, r["subject_id"])}) --> "{r["object_text"]}" ({_etype(by_id, r["object_id"])})'
              for j, r in enumerate(groups[ph], 1)]
        blocks.append("\n".join(b))
    user = f"You are given {len(phrases)} GROUPS from one document.\n\n" + "\n\n".join(blocks)
    parsed  = parse_json(gemini_json(PROPERTY_SELECT_PROMPT, user))
    results = parsed.get("results", parsed) if isinstance(parsed, dict) else parsed
    raw = {r.get("relation_phrase"): r for r in (results or []) if r.get("relation_phrase") in set(phrases)}

    out = {}
    for i, ph in enumerate(phrases):
        n   = len(groups[ph])
        top = cands[i][0] if cands[i] else {"source": "schema.org", "label": "relatedTo"}
        fallback = {"source": top["source"], "label": top["label"], "is_new": False,
                    "direction": "as_is", "justification": "fallback: top candidate"}
        asgs, seen = [], set()
        for a in ((raw.get(ph) or {}).get("assignments") or []):
            ch, idxs = (a.get("chosen_property") or {}), []
            for x in (a.get("instances") or []):
                try:
                    j = int(x) - 1
                except (TypeError, ValueError):
                    continue
                if 0 <= j < n and j not in seen:          # valid, first assignment wins on any overlap
                    seen.add(j); idxs.append(j)
            if idxs:
                asgs.append({"instances": idxs, "chosen": ch})
        missing = [j for j in range(n) if j not in seen]  # any uncovered instance -> fallback assignment
        if missing:
            asgs.append({"instances": missing, "chosen": fallback})
        out[ph] = asgs or [{"instances": list(range(n)), "chosen": fallback}]
    return out

# ---- Minting: admit a chosen "new" property into the vocabulary -------------------------------
def _schema_ref(label):
    return {"label": label, "source": "schema.org", "vocab_id": None,
            "definition": None, "parent": None, "domain": None, "range": None, "is_new_to_vocab": False}

def _vocab_ref(ent, is_new=False):
    return {"label": ent["canonical_label"], "source": "vocab", "vocab_id": ent["id"],
            "definition": ent["definition"], "parent": ent["parent"],
            "domain": ent.get("domain"), "range": ent.get("range"), "is_new_to_vocab": is_new}

# Admit a minted property as a NEW vocabulary entry (or, in restrict mode, fall back to its parent /
# a generic related-to link). Parent (subPropertyOf) may be null — properties have no single root.
def _apply_mint(aid, phrase, ch, group, vocab):
    label = ch.get("label"); definition = ch.get("definition") or ""
    parent = ch.get("parent")
    if parent is not None and not (parent in SCHEMA_LABELS or _is_vocab_label(vocab, parent)):
        parent = None                                          # unknown parent -> no super-property
    if not EXPAND_MODE:                                        # restrict mode: fall back to the parent
        return _schema_ref(parent) if parent in SCHEMA_LABELS else _schema_ref("relatedTo")
    ent = {"id": f"P{vocab['next_id']:04d}", "canonical_label": label, "definition": definition,
           "parent": parent, "domain": ch.get("domain") or [], "range": ch.get("range") or [],
           "aliases": [], "support": len(group), "first_seen": aid,
           "provenance": [{"article": aid, "relation_phrase": phrase,
                           "instances": [f'{i["subject_text"]} -> {i["object_text"]}' for i in group]}]}
    vocab["properties"].append(ent); vocab["next_id"] += 1
    return _vocab_ref(ent, is_new=True)

In [12]:
# Build canonical relation records from per-instance assignments. `inst_assignments` is a flat list of
# (phrase, instance, tref) — each instance carries the property (tref) chosen for ITS assignment, so two
# instances of the same phrase can land on different properties. Each instance is ORIENTED to its
# property's direction (subject/object SWAPPED when tref direction == "inverse"), then instances are
# DEDUPED deterministically on (subject_id, property, object_id): identical triples (e.g. a direct and an
# inverse phrase that resolve to the same property between the same two entities) collapse into ONE
# record, evidence unioned. No LLM here.
def _build_relation_records(by_id, inst_assignments):
    records = {}   # (subject_id, property_label, object_id) -> record
    for ph, r, tref in inst_assignments:
        flip = tref.get("direction") == "inverse"
        if flip:                                                  # swap to the property's direction
            s_id, s_txt, o_id, o_txt = r["object_id"], r["object_text"], r["subject_id"], r["subject_text"]
        else:
            s_id, s_txt, o_id, o_txt = r["subject_id"], r["subject_text"], r["object_id"], r["object_text"]
        key = (s_id, tref.get("label"), o_id)
        rec = records.get(key)
        if rec is None:
            rec = {
                "subject_id":   s_id,
                "subject_text": s_txt,
                "subject_type": _etype(by_id, s_id),
                "property":                 tref.get("label"),
                "property_source":          tref.get("source"),
                "vocab_id":                 tref.get("vocab_id"),
                "property_is_new_to_vocab": tref.get("is_new_to_vocab", False),
                "property_definition":      tref.get("definition"),
                "property_parent":          tref.get("parent"),
                "property_justification":   tref.get("justification"),
                "object_id":    o_id,
                "object_text":  o_txt,
                "object_type":  _etype(by_id, o_id),
                "relation_phrases": [],          # the free-form phrase(s) that produced this triple
                "phrase_directions": {},         # phrase -> "as_is" | "inverse" (was it flipped?)
                "evidence":         [],
                "confidence":       0.0,
            }
            records[key] = rec
        if ph not in rec["relation_phrases"]:
            rec["relation_phrases"].append(ph)
        rec["phrase_directions"][ph] = "inverse" if flip else "as_is"
        ev = r.get("evidence")
        if ev and ev not in rec["evidence"]:
            rec["evidence"].append(ev)
        rec["confidence"] = max(rec["confidence"], float(r.get("confidence") or 0.0))
    return list(records.values())

In [13]:
# Process ONE article end-to-end, self-contained:
#   retrieve (Hook A) -> ONE LLM call assigning each instance a property (a phrase may split across
#   several properties) -> admit any minted properties as new vocabulary entries -> build canonical
#   (deduped) relation records. Writes nary_graph_ext.json AND nary_relation_matching_review.json (per
#   relation phrase: query text, the candidates offered, and EACH assignment's instance subset + chosen property).
def process_article(aid, vocab, write=True, save=True, verbose=True):
    oie, by_id, rinfo, phrases, qtexts, groups = load_article(aid)
    cands  = retrieve_candidates(qtexts, vocab)
    report = llm_assign(phrases, rinfo, groups, by_id, cands)   # phrase -> [assignments]

    assigns = {}                                 # phrase -> [{"instances": [r...], "tref": tref, "chosen": ch}]
    inst_res, n_mint, n_flip = [], 0, 0
    for ph in phrases:
        rows = []
        for a in report[ph]:
            ch    = a["chosen"]
            insts = [groups[ph][j] for j in a["instances"]]
            is_mint  = ch.get("is_new") or ch.get("source") == "new"
            existing = _find_vocab_by_label(vocab, ch.get("label"))   # reuse an existing entry if the label matches
            # a mint is in the phrase's own direction; a reused candidate may be flipped to inverse
            direction = "as_is" if (is_mint and not existing) else ("inverse" if ch.get("direction") == "inverse" else "as_is")
            if is_mint and not existing:                              # admit a genuinely new property
                before = len(vocab["properties"])
                tref = _apply_mint(aid, ph, ch, insts, vocab)
                n_mint += (len(vocab["properties"]) > before)
            elif existing:                                            # chose (or minted a dup of) an existing vocab property
                _bump(existing, aid, ph, insts, flip=(direction == "inverse")); tref = _vocab_ref(existing)
            else:                                                     # a schema.org candidate
                tref = _schema_ref(ch.get("label"))
            tref["direction"] = direction
            tref["justification"] = ch.get("justification")
            n_flip += (direction == "inverse") * len(insts)
            for r in insts:
                inst_res.append((ph, r, tref))
            rows.append({"instances": insts, "tref": tref, "chosen": ch})
        assigns[ph] = rows

    records = _build_relation_records(by_id, inst_res)

    if write:
        out_dir = RESULTS_DIR / aid
        out_dir.mkdir(parents=True, exist_ok=True)
        (out_dir / "two_staged/3_nary_graph/nary_graph_extended.json").write_text(json.dumps(
            {"article_id": aid, "match_model": MATCH_MODEL, "expand_mode": EXPAND_MODE,
             "entities": oie.get("entities", []), "nary_nodes": oie.get("nary_nodes", []),
             "relations": records},
            indent=2, ensure_ascii=False))
        # Review file: one row per relation phrase, listing EACH assignment (its instance subset + the
        # property chosen for it, with super-property path) so a phrase split across properties is fully
        # auditable, plus the candidates the LLM chose from at THIS article's vocab state.
        def _chosen_block(tref):
            return {"label": tref.get("label"), "source": tref.get("source"), "direction": tref.get("direction"),
                    "vocab_id": tref.get("vocab_id"), "is_new_to_vocab": tref.get("is_new_to_vocab", False),
                    "definition": tref.get("definition"), "parent": tref.get("parent"),
                    "domain": tref.get("domain"), "range": tref.get("range"),
                    "path": _full_path(tref, vocab), "justification": tref.get("justification")}
        review = [{
            "relation_phrase":      ph,
            "definition":           rinfo.get(ph, {}).get("definition"),
            "embedding_query_text": qtexts[i],
            "observed_signatures":  group_signatures(groups[ph], by_id),
            "n_assignments":        len(assigns[ph]),
            "assignments": [{
                "instances":       [f'{r["subject_text"]} -> {r["object_text"]}' for r in a["instances"]],
                "chosen_property": _chosen_block(a["tref"]),
                "llm_proposed":    {k: (a["chosen"] or {}).get(k) for k in ("source", "label", "is_new", "direction")},
            } for a in assigns[ph]],
            "candidates":           cands[i],
        } for i, ph in enumerate(phrases)]
        (out_dir / "two_staged/3_nary_graph/nary_relation_matching_review.json").write_text(json.dumps(
            {"article_id": aid, "match_model": MATCH_MODEL, "expand_mode": EXPAND_MODE, "relations": review},
            indent=2, ensure_ascii=False))
    if save:
        save_vocab(vocab)
    if verbose:
        merged  = sum(1 for r in records if len(r["relation_phrases"]) > 1 or len(r["evidence"]) > 1)
        n_split = sum(1 for ph in phrases if len(assigns[ph]) > 1)
        n_inst  = len(oie["relations"])
        print(f"  {aid}: {len(phrases)} phrases ({n_split} split), {n_inst} instances -> {len(records)} triples "
              f"({merged} merged, {n_flip} inverse-reused) | +{n_mint} minted | vocab now {len(vocab['properties'])}")
    return records, vocab

## Dry run — one article, no writes

In [14]:
# # Dry run on the FIRST article against a COPY of the current vocab (no writes, no vocab mutation).
# _vocab_preview = copy.deepcopy(load_vocab())
# _records, _ = process_article(ARTICLE_ORDER[0], _vocab_preview, write=False, save=False)
# for (src, lbl), c in Counter((r.get("property_source"), r["property"]) for r in _records).most_common():
#     print(f"  {str(src):<11} {lbl}  x{c}")

## Build — incremental vocabulary over all articles (in order)

In [15]:
# Incremental build. FRESH=True: start from an EMPTY vocab and process ALL of ARTICLE_ORDER (reproducible).
# FRESH=False: keep the existing relation_vocabulary.json and process only NEW_ARTICLES (future growth).
FRESH        = True
NEW_ARTICLES = []          # used only when FRESH=False

if FRESH:
    vocab = {"properties": [], "next_id": 1}; save_vocab(vocab); todo = ARTICLE_ORDER
else:
    vocab = load_vocab()
    todo  = NEW_ARTICLES or [a for a in ARTICLE_ORDER
                             if not (RESULTS_DIR / a / "two_staged/3_nary_graph/nary_graph_extended.json").exists()]

print(f"building property vocabulary over {len(todo)} article(s): {todo}\n")
import time
nb_parallel.reset_usage()          # the report below covers only the calls THIS run makes
_t0 = time.perf_counter()

for aid in todo:
    with nb_parallel.usage_scope(aid):      # tag this article's calls so the report breaks down per article
        process_article(aid, vocab, write=True, save=True, verbose=True)
print(f"\ndone — {len(vocab['properties'])} canonical properties in {VOCAB_PATH}")

# --- what this run cost: calls, prompt / output / THINKING tokens, and elapsed time ------------
nb_parallel.usage_report("nb7b · n-ary property extension", wall_seconds=time.perf_counter() - _t0,
                         write=RESULTS_DIR / "_global" / "token_usage_nb7b_nary_property.json")

building property vocabulary over 4 article(s): ['art1', 'art2', 'art4', 'art5']

  art1: 44 phrases (5 split), 126 instances -> 126 triples (0 merged, 6 inverse-reused) | +14 minted | vocab now 14
  art2: 76 phrases (0 split), 148 instances -> 145 triples (3 merged, 12 inverse-reused) | +29 minted | vocab now 43
  art4: 19 phrases (2 split), 67 instances -> 67 triples (0 merged, 1 inverse-reused) | +7 minted | vocab now 50
  art5: 36 phrases (0 split), 97 instances -> 97 triples (0 merged, 0 inverse-reused) | +9 minted | vocab now 59

done — 59 canonical properties in /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/_global/nary_relation_vocabulary.json


## Report + verify

In [16]:
# Property vocabulary summary, then verify each ext file covers every nb6 relation instance
# (each oie relation maps to exactly one canonical triple — none lost; merges only collapse duplicates).
vocab = load_vocab()
print(f"PROPERTY VOCABULARY — {len(vocab['properties'])} canonical properties\n" + "-" * 74)
for t in sorted(vocab["properties"], key=lambda x: -x.get("support", 0)):
    al = f"  aka {t['aliases']}" if t.get("aliases") else ""
    print(f"  {t['id']}  {t['canonical_label']:<26} subPropertyOf {str(t.get('parent')):<16} "
          f"support={t.get('support')} | {_sig(t.get('domain'), t.get('range'))}{al}")

print("\nVERIFY ext vs oie_relations (every relation instance represented)\n" + "-" * 74)
ok = True
for aid in ARTICLE_ORDER:
    ext = RESULTS_DIR / aid / "two_staged/3_nary_graph/nary_graph_extended.json"
    if not ext.exists():
        print(f"  {aid:<8} (no ext file)"); continue
    oie = json.loads((RESULTS_DIR / aid / "two_staged/3_nary_graph/nary_graph.json").read_text())
    x   = json.loads(ext.read_text())["relations"]
    # every entity pair from oie must appear in some record. Compare UNORDERED pairs, since an inverse
    # reuse legitimately swaps subject/object — the pair is preserved, only its orientation changes.
    orig_pairs = {frozenset((r["subject_id"], r["object_id"])) for r in oie["relations"]}
    got_pairs  = {frozenset((r["subject_id"], r["object_id"])) for r in x}
    nvoc = sum(1 for r in x if r.get("property_source") == "vocab")
    nflip = sum(1 for r in x if "inverse" in (r.get("phrase_directions") or {}).values())
    covered = orig_pairs <= got_pairs
    ok &= covered
    print(f"  {aid:<8} {len(oie['relations']):>4} instances -> {len(x):>4} triples "
          f"({nvoc} via vocab, {nflip} inverse-reused)   {'OK' if covered else 'FAIL: pair(s) dropped'}")
print("-" * 74); print("consistent" if ok else "inconsistencies")

PROPERTY VOCABULARY — 59 canonical properties
--------------------------------------------------------------------------
  P0003  occurredDuring             subPropertyOf None             support=10 | Excess Deaths Report → Duration/Heatwave
  P0048  forCompetitionRank         subPropertyOf None             support=8 | Prize Money per Round → CompetitionRank
  P0023  isA                        subPropertyOf None             support=7 | Person/EventVenue/Festival → None
  P0045  hasPrizeMoney              subPropertyOf None             support=7 | Grand Slam Prize Money/Prize Money Award/Mixed Doubles Win/Prize Money → MonetaryAmount
  P0002  hasIncidentCount           subPropertyOf None             support=6 | Wildfire Incident Report → IncidentCount
  P0004  hasDayOfWeek               subPropertyOf None             support=6 | Red Alert Issuance/Wildfire Evacuation → DayOfWeek
  P0001  hasPopulationCount         subPropertyOf None             support=4 | Drowning Deaths Report/Wildfir

*(Optional) the GLOBAL re-consolidation and property-hierarchy passes from nb7 (cells after this point there) port over unchanged — they operate on `nary_relation_vocabulary.json` + the `nary_graph_ext.json` files. Copy them in when you want to collapse synonymous mints or print the property tree.*